# **LAB Deep Learning - Session 9 (GAN)**

## **Import Library**

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import load_img, img_to_array

## **Settings/Global Variable**

In [2]:
IMAGE_FOLDER = "./sign_mnist"
BATCH_SIZE = 32
EPOCHS = 5
NOISE_DIM = 100 # karena GAN ambil noise secara acak dari dataset, kasih batasan noise nya

seed = tf.random.normal([16, NOISE_DIM]) # 16 gambar noise random, noise nya itu2 aja, setelah 16 noisenya akan random

## **Ambil Dataset**

In [3]:
images = []

for split in ["train", "test"]:
    split_path = os.path.join(IMAGE_FOLDER, split) # masuk folder train atau test
    for kelas in os.listdir(split_path):
        kelas_path = os.path.join(split_path, kelas) # masuk folder kelas
        # masuk ke class folder(0, 1, 2, .. 24)
        for nama_file in os.listdir(kelas_path):
            img = load_img(os.path.join(kelas_path, nama_file), target_size=(28, 28), color_mode="grayscale") # load gambar. target size dari properties gambar details nya 28x28

            images.append(img_to_array(img)) # ubah gambar ke array, karena gambar itu 2D, jadi array nya 2D juga

images = np.array(images, dtype="float32") # ubah list images ke array, biar bisa diolah pake numpy

# di GAN gabisa langsung bagi ke 255. jadi -1 / 1. kalo langsung bagi ke 255, jadi semuanya true. diskriminator perlu nilai negatif. makanya pake rumus dibawah ini:
images = (images - 127.5) / 127.5 # ubah range pixel dari 0-255 ke -1 sampai 1
print("Jumlah gambar:", images.shape)

dataset = tf.data.Dataset.from_tensor_slices(images).shuffle(len(images)).batch(BATCH_SIZE) # shuffle biar random, buffer size 1024 biar randomnya lebih bagus, batch size 32 biar bisa diolah per batch

Jumlah gambar: (34627, 28, 28, 1)


## **Generator Model**

In [ ]:
def make_generator_model(): # perbesar image
    return tf.keras.Sequential([
        layers.Dense(7*7*256, use_bias=False, input_shape=(NOISE_DIM,)), # layers 12.000 an ini
        layers.BatchNormalization(), # biar lebih stabil, biar ga terlalu random
        layers.LeakyReLU(), # biar ga terlalu random, biar lebih stabil
        layers.Reshape((7, 7, 256)), # ubah ke 7x7x256

        layers.Conv2DTranspose(128, 5, strides=2, padding="same", use_bias=False), #128 perceptron
        layers.BatchNormalization(),
        layers.LeakyReLU(),

        layers.Conv2DTranspose(64, 5, strides=2, padding="same", use_bias=False), #128 perceptron
        layers.BatchNormalization(),
        layers.LeakyReLU(),

        layers.Conv2DTranspose(1, 5, strides=2, padding="same", activation="tanh")
    ])

generator = make_generator_model()
generator.summary()

c:\SEM 5\LAB AI Semester 5\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 12544)          │     1,254,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 12544)          │        50,176 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu (LeakyReLU)         │ (None, 12544)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 7, 7, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose                │ (None, 14, 14, 128)    │       819,200 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 14, 14, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_1 (LeakyReLU)       │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_1              │ (None, 28, 28, 64)     │       204,800 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 28, 28, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_2 (LeakyReLU)       │ (None, 28, 28, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_2              │ (None, 56, 56, 1)      │         1,601 │
│ (Conv2DTranspose)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,330,945 (8.89 MB)

 Trainable params: 2,305,473 (8.79 MB)

 Non-trainable params: 25,472 (99.50 KB)

## **Model Discriminator**

In [ ]:
def make_discriminator(): # kecilin image
    return tf.keras.Sequential([
        layers.Conv2D(64, 5, strides=2, padding="same", input_shape=(28, 28, 1)), # 64 perceptron
        layers.LeakyReLU(),
        layers.Dropout(0.3), # biar ga terlalu random

        layers.Conv2D(128, 5, strides=2, padding="same"), # 128 perceptron
        layers.LeakyReLU(),
        layers.Dropout(0.3),

        layers.Flatten(), # ubah ke 1D
        layers.Dense(1) # outputnya 1
    ])

discriminator = make_discriminator()
discriminator.summary()